<a href="https://colab.research.google.com/github/RenatGreen-flag/Model-Liniar-Regression/blob/main/goldPricingPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [104]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

TROY_OUNCE_TO_GRAM = 31.1035   # 1 troy ounce = 31.1035 gram

ticker_map = {
    "GC=F"    : "gold",
    "CL=F"    : "oil",
    "^JKSE"   : "ihsg",
    "^GSPC"   : "snp500",
    "^TNX"    : "treasury_us",   # Hapus tanda '-' agar aman sebagai nama kolom
    "USDIDR=X": "usdidr",
}

START = "2023-01-01"
END   = datetime.today().strftime("%Y-%m-%d")

LAG_STEPS       = [1, 3, 7]
ROLLING_WINDOWS = [3, 7]

In [105]:
print("=" *55)
print("TAHAP 1: Mengunduh data dari Yahoo Finance...")
print("=" * 55)

raw = yf.download(
    tickers     = list(ticker_map.keys()),
    start       = START,
    end         = END,
    auto_adjust = True,
    progress    = True
)

df = raw["Close"].rename(columns=ticker_map).add_suffix("_close")
print(f"\nShape setelah download: {df.shape}")
print(f"Rentang tanggal awal  : {df.index.min().date()} s.d. {df.index.max().date()}")


[*********************100%***********************]  6 of 6 completed

TAHAP 1: Mengunduh data dari Yahoo Finance...

Shape setelah download: (899, 6)
Rentang tanggal awal  : 2023-01-02 s.d. 2026-06-17


In [106]:
# TAHAP 2: PENYELARASAN TIMEZONE (SHIFT +1 HARI)
GLOBAL_TICKERS = list(ticker_map.keys())

for ticker_name in GLOBAL_TICKERS:
  col = f"{ticker_name}_close"
  if col in df.columns:
    df[col] = df[col].shift(1)


In [107]:
# TAHAP 3: REINDEX KE KALENDER HARIAN PENUH

full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
df = df.reindex(full_range)
df.index.name = "Date"

print(f"Shape telah reindex: {df.shape}")


Shape telah reindex: (1263, 6)


In [108]:
# TAHAP 4: FORWARD FILL (FFILL) -> HARGA CLOSE HARI TERAKHIR

price_cols = [c for c in df.columns if c.endswith("_close")]
df[price_cols] = df[price_cols].ffill().bfill()

missing_after = df[price_cols].isnull().sum()
print(f"\nJumlah missing value setelah forward fill:\n{missing_after}")



Jumlah missing value setelah forward fill:
Ticker
oil_close            0
gold_close           0
usdidr_close         0
snp500_close         0
ihsg_close           0
treasury_us_close    0
dtype: int64


In [109]:
# # feature engineering gold usd -> idr per gram
df["gold_close_idr_per_oz"] = df["gold_close"] * df['usdidr_close']

df["gold_close_idr_per_gram"] = df["gold_close_idr_per_oz"] / TROY_OUNCE_TO_GRAM

price_cols = price_cols + ["gold_close_idr_per_oz","gold_close_idr_per_gram"]
print(df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]].tail(3).to_string())

Ticker       gold_close  usdidr_close  gold_close_idr_per_oz  gold_close_idr_per_gram
Date                                                                                 
2026-06-15  4328.000000       17772.0           7.691722e+07             2.472944e+06
2026-06-16  4330.899902       17716.0           7.672622e+07             2.466804e+06
2026-06-17  4358.899902       17730.0           7.728330e+07             2.484714e+06


In [110]:
# # feature engineering : lag feature (H-1, H-3, H-7)

for col in price_cols:
  for lag in LAG_STEPS:
    df[f"{col}_lag{lag}"] = df[col].shift(lag)

lag_cols = [c for c in df.columns if "gold" in c and "lag" in c]
print(f"Contoh kolom lag gold: {lag_cols}")

Contoh kolom lag gold: ['gold_close_lag1', 'gold_close_lag3', 'gold_close_lag7', 'gold_close_idr_per_oz_lag1', 'gold_close_idr_per_oz_lag3', 'gold_close_idr_per_oz_lag7', 'gold_close_idr_per_gram_lag1', 'gold_close_idr_per_gram_lag3', 'gold_close_idr_per_gram_lag7']


In [111]:
# # feature engineering: Rolling statistic per time step

for col in price_cols:
  for window in ROLLING_WINDOWS:
    df[f"{col}_rmean_{window}"] = df[col].rolling(window=window).mean()
    df[f"{col}_rstd_{window}"] = df[col].rolling(window=window).std()
    df[f"{col}_rmin_{window}"] = df[col].rolling(window=window).min()
    df[f"{col}_rmax_{window}"] = df[col].rolling(window=window).max()

In [112]:
# # Featur engineering: persentase perubahan (RETURN HARIAN)
for col in price_cols:
  df[f"{col}_pct_change"] = df[col].pct_change()

df_clean = df.dropna().reset_index()
print(f"\nShape FINAL setelah semua tahap preprocessing: {df_clean.shape}")
print(f"Rentang tanggal final : {df_clean['Date'].min().date()} s.d. {df_clean['Date'].max().date()}")


Shape FINAL setelah semua tahap preprocessing: (1256, 105)
Rentang tanggal final : 2023-01-09 s.d. 2026-06-17


/tmp/ipykernel_2184/1473989071.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_pct_change"] = df[col].pct_change()
/tmp/ipykernel_2184/1473989071.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_pct_change"] = df[col].pct_change()
/tmp/ipykernel_2184/1473989071.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

In [113]:
# # TAHAP 10: SIMPAN KE CSV + RINGKASAN DATASET

output_path = "gold_preprocessed_final.csv"
df_clean.to_csv(output_path, index=False)

In [114]:
print("\n" + "=" * 55)
print("RINGKASAN DATASET FINAL")
print("=" * 55)

# Pengelompokan kolom berdasarkan tipe fitur
cols_raw     = [c for c in df_clean.columns if c.endswith("_close") and "lag" not in c and "rmean" not in c]
cols_derived = ["gold_idr_per_oz", "gold_idr_per_gram"]
cols_lag     = [c for c in df_clean.columns if "_lag" in c]
cols_rolling = [c for c in df_clean.columns if any(x in c for x in ["_rmean_", "_rstd_", "_rmin_", "_rmax_"])]
cols_return  = [c for c in df_clean.columns if "_pct_change" in c]

print(f"\n{'Tipe Fitur':<25} {'Jumlah Kolom':>13}")
print("-" * 40)
print(f"{'Kolom mentah (_close)':<25} {len(cols_raw):>13}")
print(f"{'Fitur turunan (IDR)':<25} {len(cols_derived):>13}")
print(f"{'Fitur lag (H-1/3/7)':<25} {len(cols_lag):>13}")
print(f"{'Rolling statistics':<25} {len(cols_rolling):>13}")
print(f"{'Return harian':<25} {len(cols_return):>13}")
print("-" * 40)
print(f"{'TOTAL KOLOM (+ Date)':<25} {df_clean.shape[1]:>13}")
print(f"{'Total baris':<25} {df_clean.shape[0]:>13}")

print(f"\nFile tersimpan ke: {output_path}")
print("\n5 baris pertama dataset:")
print(df_clean[["Date", "gold_close", "usdidr_close", "gold_close_idr_per_gram",
                 "gold_close_lag1", "gold_close_lag7",
                 "gold_close_rmean_7", "gold_close_pct_change"]].head().to_string(index=False))


RINGKASAN DATASET FINAL

Tipe Fitur                 Jumlah Kolom
----------------------------------------
Kolom mentah (_close)                 6
Fitur turunan (IDR)                   2
Fitur lag (H-1/3/7)                  24
Rolling statistics                   64
Return harian                         8
----------------------------------------
TOTAL KOLOM (+ Date)                105
Total baris                        1256

File tersimpan ke: gold_preprocessed_final.csv

5 baris pertama dataset:
      Date  gold_close  usdidr_close  gold_close_idr_per_gram  gold_close_lag1  gold_close_lag7  gold_close_rmean_7  gold_close_pct_change
2023-01-09 1872.699951  15602.299805            939393.511419      1864.199951      1839.699951         1856.085693               0.004560
2023-01-10 1871.599976  15580.000000            937499.883281      1872.699951      1839.699951         1860.642840              -0.000587
2023-01-11 1874.599976  15528.900391            935922.847691      1871.599976   